# GREG walkthrough: linear calibration via R's `survey` package

This notebook walks through a single end-to-end GREG calibration run on a ten-household
synthetic survey. The goal is to show the exact input and output formats the GREG runner
consumes so that the paper's benchmark scaffold is easy to audit.

GREG (Generalized Regression Estimator) is a classical survey-calibration method. It
rescales each unit's design weight so that, taken together, the weighted sample exactly
reproduces a set of known population totals. Its natural input is a linear system:

$$
X\, w_{\text{fitted}} = t
$$

where `X` is a `(n_targets, n_units)` matrix of per-unit coefficients, `t` is the target
vector, and `w_{fitted}` is the fitted per-unit weight. GREG finds `w_{fitted}` that hits
`t` exactly (when feasible) while staying as close as possible to the design weights under
a chi-squared-style distance.

In [1]:
import subprocess
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import mmwrite
from scipy.sparse import csr_matrix

from toy_data import (
    HOUSEHOLDS,
    TARGETS,
    build_target_matrix,
    diagnostic_table,
    household_table_with_coefficients,
)

NOTEBOOK_DIR = Path().resolve()
GREG_RUNNER = NOTEBOOK_DIR.parent / "runners" / "greg_runner.R"
assert GREG_RUNNER.exists(), GREG_RUNNER

## 1. The toy survey

Ten households split across two districts (`A` and `B`). Each household has a design
weight of `100`, a household size, an adult/child breakdown, and a household income.
The design weights, summed, represent 1000 households nationally.

In [2]:
HOUSEHOLDS

,hh_id,district,n_adults,n_children,income,design_weight,hh_size
0,1,A,2,0,30000,100.0,2
1,2,A,2,2,60000,100.0,4
2,3,A,1,0,25000,100.0,1
3,4,A,2,1,45000,100.0,3
4,5,A,2,0,70000,100.0,2
5,6,B,2,3,40000,100.0,5
6,7,B,1,1,20000,100.0,2
7,8,B,2,1,80000,100.0,3
8,9,B,3,1,90000,100.0,4
9,10,B,1,0,15000,100.0,1


## 2. Targets

We want to calibrate to eight administrative totals: four per district covering household
count, adults, children, and total household income. The baseline column shows where the
design weights land before calibration.

In [3]:
hh = household_table_with_coefficients()
base_w = hh["design_weight"].to_numpy(dtype=np.float64)
X = build_target_matrix(hh, TARGETS)

baseline_df = pd.DataFrame(
    {
        "target_name": TARGETS["target_name"],
        "target_value": TARGETS["value"].astype(float),
        "baseline_weighted_total": X @ base_w,
    }
)
baseline_df

,target_name,target_value,baseline_weighted_total
0,district_A_households,520.0,500.0
1,district_A_adults,940.0,900.0
2,district_A_children,310.0,300.0
3,district_A_income,23500000.0,23000000.0
4,district_B_households,480.0,500.0
5,district_B_adults,1000.0,900.0
6,district_B_children,600.0,600.0
7,district_B_income,25000000.0,24500000.0


## 3. Input format 1/3 — the `(n_targets, n_units)` matrix `X`

Each row of `X` is one target; each column is one household. Entry `X[t, i]` is "how
much household `i` would contribute to target `t` if its fitted weight were `1`".

In the benchmark scaffold this matrix is held in [Matrix Market](https://math.nist.gov/MatrixMarket/)
format (`.mtx`) so the R runner can read it sparsely via `Matrix::readMM`.

In [4]:
X_df = pd.DataFrame(
    X,
    index=TARGETS["target_name"],
    columns=[f"hh_{i}" for i in HOUSEHOLDS["hh_id"]],
)
X_df

,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,hh_8,hh_9,hh_10
target_name,,,,,,,,,,
district_A_households,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
district_A_adults,2.0,2.0,1.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0
district_A_children,0.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
district_A_income,30000.0,60000.0,25000.0,45000.0,70000.0,0.0,0.0,0.0,0.0,0.0
district_B_households,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
district_B_adults,0.0,0.0,0.0,0.0,0.0,2.0,1.0,2.0,3.0,1.0
district_B_children,0.0,0.0,0.0,0.0,0.0,3.0,1.0,1.0,1.0,0.0
district_B_income,0.0,0.0,0.0,0.0,0.0,40000.0,20000.0,80000.0,90000.0,15000.0


## 4. Input format 2/3 — target metadata

A CSV with one row per target. GREG only needs the `value` column; the `target_name`
column is carried through for diagnostics. In the production benchmark this file also
carries geography and target-family columns used by the scaffold's filtering logic.

In [5]:
target_metadata = pd.DataFrame(
    {
        "target_name": TARGETS["target_name"],
        "value": TARGETS["value"].astype(float),
    }
)
target_metadata

,target_name,value
0,district_A_households,520.0
1,district_A_adults,940.0
2,district_A_children,310.0
3,district_A_income,23500000.0
4,district_B_households,480.0
5,district_B_adults,1000.0
6,district_B_children,600.0
7,district_B_income,25000000.0


## 5. Input format 3/3 — initial weights

A plain `.npy` array with one entry per unit. GREG will scale each entry up or down to
meet the targets.

In [6]:
initial_weights = HOUSEHOLDS["design_weight"].to_numpy(dtype=np.float64)
initial_weights

array([100., 100., 100., 100., 100., 100., 100., 100., 100., 100.])

## 6. Write the bundle and invoke the R runner

The R runner lives at `paper-l0/benchmarking/runners/greg_runner.R`. It is a thin wrapper
around `survey::grake` with `cal.linear` (classical GREG). We call it as a subprocess so
this notebook can be run from Python.

In [7]:
tmp = Path(tempfile.mkdtemp(prefix="greg_toy_"))

mmwrite(str(tmp / "X_targets_by_units.mtx"), csr_matrix(X))
target_metadata.to_csv(tmp / "target_metadata.csv", index=False)
np.save(tmp / "initial_weights.npy", initial_weights)

out_csv = tmp / "fitted_weights.csv"

cmd = [
    "Rscript",
    str(GREG_RUNNER),
    str(tmp / "X_targets_by_units.mtx"),
    str(tmp / "target_metadata.csv"),
    str(tmp / "initial_weights.npy"),
    str(out_csv),
    "200",  # maxit
    "1e-7",  # epsilon
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print("return code:", proc.returncode)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
sorted(p.name for p in tmp.iterdir())

return code: 0


['X_targets_by_units.mtx',
 'fitted_weights.csv',
 'initial_weights.npy',
 'target_metadata.csv']

## 7. Output — fitted weights

The runner writes a two-column CSV (`unit_index`, `fitted_weight`). The benchmark CLI
immediately converts that to a `.npy` at the per-unit level; we do the same here for
symmetry with the IPF notebook.

In [8]:
fitted_df = pd.read_csv(out_csv)
fitted_weights = fitted_df["fitted_weight"].to_numpy(dtype=np.float64)
weights_view = pd.DataFrame(
    {
        "hh_id": HOUSEHOLDS["hh_id"],
        "district": HOUSEHOLDS["district"],
        "design_weight": initial_weights,
        "fitted_weight": fitted_weights,
        "ratio_to_design": fitted_weights / initial_weights,
    }
)
weights_view

,hh_id,district,design_weight,fitted_weight,ratio_to_design
0,1,A,100.0,116.875,1.16875
1,2,A,100.0,100.625,1.00625
2,3,A,100.0,100.000,1.00000
3,4,A,100.0,108.750,1.08750
4,5,A,100.0,93.750,0.93750
5,6,B,100.0,120.500,1.20500
6,7,B,100.0,46.500,0.46500
7,8,B,100.0,-15.500,-0.15500
8,9,B,100.0,207.500,2.07500
9,10,B,100.0,121.000,1.21000


## 8. Did we hit the targets?

Compare the weighted totals under the fitted weights against the declared target values.

In [9]:
diagnostic_table(hh, TARGETS, fitted_weights).style.format(
    {
        "target_value": "{:,.1f}",
        "baseline_weighted_total": "{:,.1f}",
        "fitted_weighted_total": "{:,.1f}",
        "abs_rel_error": "{:.2e}",
    }
)

,target_name,target_value,baseline_weighted_total,fitted_weighted_total,abs_rel_error
0,district_A_households,520.0,500.0,520.0,3.61e-13
1,district_A_adults,940.0,900.0,940.0,3.23e-13
2,district_A_children,310.0,300.0,310.0,1.26e-13
3,district_A_income,"23,500,000.0","23,000,000.0","23,500,000.0",3.52e-14
4,district_B_households,480.0,500.0,480.0,5.65e-11
5,district_B_adults,"1,000.0",900.0,"1,000.0",2.09e-11
6,district_B_children,600.0,600.0,600.0,2.18e-11
7,district_B_income,"25,000,000.0","24,500,000.0","25,000,000.0",2.07e-11


## 9. Known pathology — negative weights

With `cal.linear`, GREG imposes no sign constraint on the fitted weights. On a small,
tightly-constrained problem like this one it can drive individual weights negative. The
`survey` package supports logit / raking variants to avoid this, but those variants
introduce their own trade-offs and do not cleanly generalize to the full subnational
calibration problem. In the paper this is discussed as one of the limitations of GREG
at production scale.

In [10]:
n_negative = int((fitted_weights < 0).sum())
print(f"Negative-weight units: {n_negative}")
weights_view[weights_view["fitted_weight"] < 0]

Negative-weight units: 1


,hh_id,district,design_weight,fitted_weight,ratio_to_design
7,8,B,100.0,-15.5,-0.155


## Summary

Inputs to the GREG runner:

| Artifact | Format | Shape |
| --- | --- | --- |
| `X_targets_by_units.mtx` | Matrix Market sparse | `(n_targets, n_units)` |
| `target_metadata.csv` | CSV with a `value` column | `n_targets` rows |
| `initial_weights.npy` | float64 array | `n_units` |

Output of the GREG runner:

| Artifact | Format | Shape |
| --- | --- | --- |
| `fitted_weights.csv` | CSV (`unit_index`, `fitted_weight`) | `n_units` rows |

Next notebook: `02_ipf_walkthrough.ipynb` — same toy data, but through the IPF runner,
which needs a very different input representation.